<a href="https://colab.research.google.com/github/VrishankDesai/Data-Science-Projects/blob/main/Project_1_Forecasting_renewable_energy_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import xgboost as xgb

In [4]:
data = pd.read_csv('/content/archive (2).zip', parse_dates=['Unnamed: 0'], index_col=['Unnamed: 0'])
data.index = pd.to_datetime(data.index)


FileNotFoundError: [Errno 2] No such file or directory: '/content/Prediction+of+Renewable+Energy+Generation.zip!Project Code/ML_Model/Turbine_Data.csv'

In [ ]:
data.head()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.nunique()

In [ ]:
data.drop(['ControlBoxTemperature','WTG'], axis=1, inplace=True)

In [ ]:
data.shape

In [ ]:
data.describe()

In [ ]:
data =  data[data['ActivePower']>=0]

In [ ]:
data.shape

In [ ]:
data.isna().sum()

In [ ]:
#data visualization
d2 = data.copy()
for i in data:
    mini = min(d2[i])
    maxi = max(d2[i])
    d2[i] = (d2[i]- mini)/(maxi-mini)

    plt.figure(figsize=(10,3))
    d2[i].resample('D').mean().plot(legend=i)
    (d2['ActivePower']*0.7).resample('D').mean().plot(style='--', legend='ActivePower')
    plt.legend()
    plt.show()

In [ ]:
for i in data:


    plt.figure(figsize=(10,3))
    d2[i].resample('M').mean().plot(legend=i)
    (d2['ActivePower']*0.7).resample('M').mean().plot(style='--', legend='ActivePower')
    plt.legend()
    plt.show()

In [ ]:
sns.heatmap(data.corr())

In [ ]:
correlations = data.corr().unstack().sort_values(ascending=False)

# checking  ActivePower correlations
correlations['ActivePower'].drop_duplicates()

In [ ]:
data = data[['ActivePower','WindSpeed']]
data.dropna()

In [ ]:
len(data)

In [ ]:
X_train, X_test, y_train,y_test = data['WindSpeed'][0:78000], data['WindSpeed'][78000:], data['ActivePower'][0:78000],  data['ActivePower'][78000:]
len(X_train),len(X_test),len(y_train),len(y_test)

In [ ]:
model = xgb.XGBRegressor(n_estimators = 200)

In [ ]:
model.fit(X_train.values.reshape(-1, 1), y_train,
          eval_set=[(X_train.values.reshape(-1, 1), y_train),
                    (X_test.values.reshape(-1, 1), y_test)],
          verbose=True)

In [ ]:
pred = model.predict(X_test)

In [ ]:
df_final = pd.DataFrame(data={'Actuals':y_test, 'Predictions':pred })

In [ ]:
from sklearn.metrics import *
import numpy as np

print('The Coefficient of determination (R-squared) = {:.3f}'.format(r2_score(df_final['Actuals'],df_final['Predictions'])))
print('The mean absolute error (MAE)                = {:.2f}'.format(mean_absolute_error(df_final['Actuals'],df_final['Predictions'])))
print('The RMSE error (RMSE)                        = {:.2f}'.format(np.sqrt(mean_squared_error(df_final['Actuals'],df_final['Predictions']))))
print('The Mean absolute percentage error (MAPE)    = {:.3f}'.format(mean_absolute_percentage_error(df_final['Actuals'],df_final['Predictions'])))

In [ ]:
plt.plot(df_final['Predictions'],label='Prediction')
plt.plot(df_final['Actuals'], color='red',label='Actual')

plt.ylabel('Active Power')
plt.legend()
plt.tight_layout()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(15,2))
sns.lineplot(data = data['ActivePower'][74000:])
sns.lineplot(data = df_final['Predictions'])

plt.figure(figsize=(15,2))
sns.lineplot(data = data['ActivePower'][74000:].resample('D').mean())
sns.lineplot(data = df_final['Predictions'].resample('D').mean())